In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_departments
# Source          : departments.csv
# Target          : procurement.bronze.bronze_departments
# Audit Table     : procurement.audit.duplicate_departments
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw department master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Department master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Department IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_DEPARTMENTS)
print(AUDIT_DUPLICATE_DEPARTMENTS)
print(DEPARTMENTS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Departments schema
departments_schema = StructType([
    StructField("department_id", StringType(), False),
    StructField("department_name", StringType(), True),
    StructField("division", StringType(), True),
    StructField("region", StringType(), True),
    StructField("cost_center", StringType(), True),
    StructField("annual_budget_usd", DecimalType(18,2), True)
])
# Read Departments master data from landing volume
bronze_departments_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(departments_schema)
    .load(DEPARTMENTS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_departments_df.count()}")

print("\nSchema:")
bronze_departments_df.printSchema()

print("\nColumns:")
print(bronze_departments_df.columns)

print("\nSampledata:")
display(bronze_departments_df.limit(10))

In [0]:
# ============================================================
# Identify Invalid Department Records
# ============================================================

invalid_departments = bronze_departments_df.filter(

    col("department_id").isNull() |
    (trim(col("department_id")) == "") |

    col("department_name").isNull() |
    (trim(col("department_name")) == "") |

    col("division").isNull() |
    (trim(col("division")) == "") |

    col("region").isNull() |
    (trim(col("region")) == "") |

    col("cost_center").isNull() |
    (trim(col("cost_center")) == "") |

    col("annual_budget_usd").isNull() |
    (col("annual_budget_usd") < 0)
)

print(f"Invalid Department Records : {invalid_departments.count()}")

display(invalid_departments)

In [0]:
# ============================================================
# Identify Duplicate Department IDs
# ============================================================

duplicate_department_keys = (
    bronze_departments_df
    .filter(
        col("department_id").isNotNull() &
        (trim(col("department_id")) != "")
    )
    .groupBy("department_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_department_keys)

In [0]:
# ============================================================
# Identify Duplicate Department Records
# Business Rule: Keep the first occurrence of each Department ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("department_id").orderBy("department_id")

department_rank_df = (
    bronze_departments_df
        .join(
            duplicate_department_keys.select("department_id"),
            on="department_id",
            how="inner"
        )
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)
display(department_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate Department Records
# ============================================================

duplicate_departments = (
    department_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Department Records : {duplicate_departments.count()}")

display(duplicate_departments)

In [0]:
# ============================================================
# Add Audit Metadata for duplicate department IDs
# ============================================================

from pyspark.sql.functions import current_timestamp, lit

duplicate_departments = (
    duplicate_departments
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Departments"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_departments)

In [0]:
# ============================================================
# Add Audit Metadata for NULL Department IDs
# ============================================================

invalid_departments = (
    null_blank_department_id
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("departments"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Invalid Record"))
)

display(invalid_departments)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_departments.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_departments,
         table_name = AUDIT_DUPLICATE_DEPARTMENTS
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_DEPARTMENTS}")

else:

    print("No duplicate Department records found. Audit table not created.")

In [0]:
# ============================================================
# Write Invalid NULL Records to Audit Table
# ============================================================

invalid_count = invalid_departments.count()

if invalid_count > 0:

    write_delta(
        df = invalid_departments,
         table_name = AUDIT_INVALID_DEPARTMENTS
    )

    print(f"Successfully written {invalid_count} invalid record(s) to {AUDIT_INVALID_DEPARTMENTS}")

else:

    print("No invalid departments records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_departments_final_df = (
    bronze_departments_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Departments.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(
    df=bronze_departments_final_df,
    table_name=BRONZE_DEPARTMENTS
)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_departments = spark.table(BRONZE_DEPARTMENTS)

print(f"Total Bronze Records : {bronze_departments.count()}")

display(bronze_departments)

In [0]:
# ============================================================
# Bronze Department complete summary
# ============================================================
print("=" * 60)
print("Bronze Departments Load Completed Successfully")
print("=" * 60)

print(f"{'Landing Records':<30}: {bronze_departments_df.count()}")

print(f"{'Duplicate Audit Records':<30}: {duplicate_departments.count()}")

print(f"{'Invalid Purchase Orders Records':<30}: {invalid_departments.count()}")

print(f"{'Total Audit Records':<30}: {duplicate_departments.count() + invalid_departments.count()}")

print(f"{'Bronze Records':<30}: {spark.table(BRONZE_DEPARTMENTS).count()}")